## Import Libraries
Imports all libraries needed for data manipulation (`numpy`, `pandas`), visualization (`matplotlib`, `seaborn`), text parsing (`re`), machine learning (`scikit-learn`, `xgboost`), model evaluation, hyperparameter tuning, and feature importance (`shap`). Warnings are suppressed and pandas is set to display all columns.

In [ ]:
# Import Libraries

import warnings
warnings.filterwarnings("ignore")

# Data Manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Regular Expressions
import re

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

# Model Evaluation
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV

# Feature Importance
import shap

# Display Settings
pd.set_option("display.max_columns", None)

## Load Dataset
Reads the mobile phone dataset from `Dataset.xlsx` into a DataFrame, prints its shape (rows, columns), and previews the first 5 rows.

In [ ]:
# Load Dataset

df = pd.read_excel("Dataset.xlsx")

print(f"Dataset Shape : {df.shape}")

df.head()

Dataset Shape : (5627, 17)


,Name,Brand,Price,MRP,Discount,Rating,Ratings,Reviews,RAM,Storage,Display,Camera,Battery,Processor,Product URL,Image URL,Processor_Clean
0,"Ai+ Nova 2 Ultra 5G (Purple, 128 GB)",ai_plus,17999,25999,30,4.1,903,88,6,128,17.22 cm (6.78 inch) Full HD+ AMOLED Display,50MP + 8MP | 13MP Front Camera,6000,Dimensity 7400 Processor,https://www.flipkart.com/ai-nova-2-ultra-5g-pu...,https://rukminim2.flixcart.com/image/312/312/x...,Dimensity 7400
1,"Ai+ Pulse 1 (Blue, 64 GB)",ai_plus,7999,7999,0,4.2,43123,2831,4,64,17.13 cm (6.745 inch) HD+ Display,50MP Rear Camera | 5MP Front Camera,5000,T615 Processor,https://www.flipkart.com/ai-pulse-1-blue-64-gb...,https://rukminim2.flixcart.com/image/312/312/x...,T615
2,"Ai+ Nova 2 Ultra 5G (Red, 128 GB)",ai_plus,19999,26999,25,4.0,413,53,8,128,17.22 cm (6.78 inch) Full HD+ AMOLED Display,50MP + 8MP | 13MP Front Camera,6000,Dimensity 7400 Processor,https://www.flipkart.com/ai-nova-2-ultra-5g-re...,https://rukminim2.flixcart.com/image/312/312/x...,Dimensity 7400
3,"Ai+ Pulse 1 (Black, 64 GB)",ai_plus,7999,7999,0,4.2,43123,2831,4,64,17.13 cm (6.745 inch) HD+ Display,50MP Rear Camera | 5MP Front Camera,5000,T615 Processor,https://www.flipkart.com/ai-pulse-1-black-64-g...,https://rukminim2.flixcart.com/image/312/312/x...,T615
4,"Ai+ Nova 2 Ultra 5G (Green, 128 GB)",ai_plus,19999,26999,25,4.0,413,53,8,128,17.22 cm (6.78 inch) Full HD+ AMOLED Display,50MP + 8MP | 13MP Front Camera,6000,Dimensity 7400 Processor,https://www.flipkart.com/ai-nova-2-ultra-5g-gr...,https://rukminim2.flixcart.com/image/312/312/x...,Dimensity 7400


## Display Size Extraction (v1)
Defines `extract_display_size`, which uses a regex to pull the screen size in inches out of the `Display` text column, and applies it to create a new `Display_Size` column. (This first version only catches sizes explicitly given in inches — it is later replaced by an improved version in a later cell.)

In [ ]:
# Display Size Extraction

def extract_display_size(display):
    match = re.search(r"\(([\d.]+)\s*inch", str(display))
    if match:
        return float(match.group(1))
    return np.nan

df["Display_Size"] = df["Display"].apply(extract_display_size)

## Display Type Extraction (v1)
Defines `extract_display_type`, which classifies the display technology (`AMOLED`, `OLED`, `LCD`, `IPS`, or `Other`) by keyword-matching the `Display` text, and applies it to create a `Display_Type` column. (A more detailed version appears later in the notebook.)

In [ ]:
# Display Type Extraction

def extract_display_type(display):

    display = str(display).upper()

    if "AMOLED" in display:
        return "AMOLED"

    elif "OLED" in display:
        return "OLED"

    elif "LCD" in display:
        return "LCD"

    elif "IPS" in display:
        return "IPS"

    else:
        return "Other"

df["Display_Type"] = df["Display"].apply(extract_display_type)

## Rear Camera Extraction (v1)
Defines `extract_rear_camera`, which splits the `Camera` text on `|` and pulls the first megapixel value from the rear-camera portion, storing it in `Rear_Main_MP`.

In [ ]:
# Rear Camera Extraction

def extract_rear_camera(camera):

    camera = str(camera)

    rear = camera.split("|")[0]

    values = re.findall(r"(\d+)MP", rear)

    if values:
        return int(values[0])

    return np.nan

df["Rear_Main_MP"] = df["Camera"].apply(extract_rear_camera)

## Front Camera Extraction (v1)
Defines `extract_front_camera`, which takes the portion of the `Camera` text after the `|` separator and extracts the front camera's megapixel value into `Front_MP`.

In [ ]:
# Front Camera Extraction

def extract_front_camera(camera):

    camera = str(camera)

    if "|" in camera:

        front = camera.split("|")[1]

        values = re.findall(r"(\d+)MP", front)

        if values:
            return int(values[0])

    return np.nan

df["Front_MP"] = df["Camera"].apply(extract_front_camera)

## Inspect Extracted Features
Displays the raw `Display`/`Camera` columns side-by-side with the newly engineered `Display_Size`, `Display_Type`, `Rear_Main_MP`, and `Front_MP` columns to sanity-check the extraction logic on the first 10 rows.

In [ ]:
df[
    [
        "Display",
        "Display_Size",
        "Display_Type",
        "Camera",
        "Rear_Main_MP",
        "Front_MP"
    ]
].head(10)

,Display,Display_Size,Display_Type,Camera,Rear_Main_MP,Front_MP
0,17.22 cm (6.78 inch) Full HD+ AMOLED Display,6.780,AMOLED,50MP + 8MP | 13MP Front Camera,50.0,13.0
1,17.13 cm (6.745 inch) HD+ Display,6.745,Other,50MP Rear Camera | 5MP Front Camera,50.0,5.0
2,17.22 cm (6.78 inch) Full HD+ AMOLED Display,6.780,AMOLED,50MP + 8MP | 13MP Front Camera,50.0,13.0
3,17.13 cm (6.745 inch) HD+ Display,6.745,Other,50MP Rear Camera | 5MP Front Camera,50.0,5.0
4,17.22 cm (6.78 inch) Full HD+ AMOLED Display,6.780,AMOLED,50MP + 8MP | 13MP Front Camera,50.0,13.0
5,17.13 cm (6.745 inch) HD+ Display,6.745,Other,50MP Rear Camera | 5MP Front Camera,50.0,5.0
6,17.22 cm (6.78 inch) Full HD+ AMOLED Display,6.780,AMOLED,50MP + 8MP | 13MP Front Camera,50.0,13.0
7,43.23 cm (17.018 cm) HD+ Display,NaN,Other,50MP Rear Camera | 5MP Front Camera,50.0,5.0
8,17.22 cm (6.78 inch) Full HD+ AMOLED Display,6.780,AMOLED,50MP + 8MP | 13MP Front Camera,50.0,13.0
9,17.22 cm (6.78 inch) Full HD+ AMOLED Display,6.780,AMOLED,50MP + 8MP | 13MP Front Camera,50.0,13.0


## Check Missing Values
Counts null values produced in each engineered column so far, to see how many rows the regex-based extractors failed to parse.

In [ ]:
df[
    [
        "Display_Size",
        "Display_Type",
        "Rear_Main_MP",
        "Front_MP"
    ]
].isnull().sum()

,0
Display_Size,279
Display_Type,0
Rear_Main_MP,86
Front_MP,1098


## Improved Display Size Extraction
Redefines `extract_display_size` to also handle cases where only a centimeter measurement is given (converting cm to inches), reducing the number of missing values compared to the first version.

In [ ]:
# ==========================================
# Improved Display Size Extraction
# ==========================================

def extract_display_size(display):
    display = str(display)

    # Case 1: Size mentioned in inches
    inch_match = re.search(r"\(([\d.]+)\s*inch", display, re.IGNORECASE)
    if inch_match:
        return float(inch_match.group(1))

    # Case 2: Only centimeters mentioned
    cm_match = re.search(r"([\d.]+)\s*cm", display, re.IGNORECASE)
    if cm_match:
        cm = float(cm_match.group(1))
        return round(cm / 2.54, 2)

    return np.nan


df["Display_Size"] = df["Display"].apply(extract_display_size)

## Rear Main Camera Extraction (v2)
Redefines `extract_rear_camera` with a regex that also captures decimal megapixel values (e.g. `12.5MP`), improving on the integer-only version used earlier.

In [ ]:
# ==========================================
# Rear Main Camera Extraction
# ==========================================

def extract_rear_camera(camera):

    camera = str(camera)

    rear = camera.split("|")[0]

    values = re.findall(r"(\d+(?:\.\d+)?)MP", rear, re.IGNORECASE)

    if values:
        return float(values[0])

    return np.nan


df["Rear_Main_MP"] = df["Camera"].apply(extract_rear_camera)

## Front Camera Extraction (v2)
Redefines `extract_front_camera` with the same decimal-value support added to the rear-camera extractor, for consistency.

In [ ]:
# ==========================================
# Front Camera Extraction
# ==========================================

def extract_front_camera(camera):

    camera = str(camera)

    if "|" in camera:

        front = camera.split("|")[1]

        values = re.findall(r"(\d+(?:\.\d+)?)MP", front, re.IGNORECASE)

        if values:
            return float(values[0])

    return np.nan


df["Front_MP"] = df["Camera"].apply(extract_front_camera)

## Rear Camera Count
Defines `rear_camera_count`, which counts how many camera sensors are listed in the rear-camera portion of the `Camera` string, and stores the count in `Rear_Camera_Count`.

In [ ]:
# ==========================================
# Rear Camera Count
# ==========================================

def rear_camera_count(camera):

    camera = str(camera)

    rear = camera.split("|")[0]

    return len(re.findall(r"(\d+(?:\.\d+)?)MP", rear))


df["Rear_Camera_Count"] = df["Camera"].apply(rear_camera_count)

## Missing Value Treatment
Fills any remaining missing values in `Display_Size`, `Rear_Main_MP`, and `Front_MP` with each column's median value.

In [ ]:
# ==========================================
# Missing Value Treatment
# ==========================================

df["Display_Size"].fillna(df["Display_Size"].median(), inplace=True)

df["Rear_Main_MP"].fillna(df["Rear_Main_MP"].median(), inplace=True)

df["Front_MP"].fillna(df["Front_MP"].median(), inplace=True)

## Engineered Features Summary
Re-checks null counts across all engineered features (`Display_Size`, `Display_Type`, `Rear_Main_MP`, `Front_MP`, `Rear_Camera_Count`) to confirm the missing-value treatment worked.

In [ ]:
# ==========================================
# Engineered Features Summary
# ==========================================

engineered_cols = [
    "Display_Size",
    "Display_Type",
    "Rear_Main_MP",
    "Front_MP",
    "Rear_Camera_Count"
]

df[engineered_cols].isnull().sum()

,0
Display_Size,0
Display_Type,0
Rear_Main_MP,0
Front_MP,0
Rear_Camera_Count,0


## Refined Display Type Extraction
A more granular version of `extract_display_type` that distinguishes `Dynamic AMOLED`, `Super AMOLED`, `AMOLED`, `P-OLED`, `OLED`, `IPS LCD`, `IPS`, `LCD`, and falls back to `LCD (Unspecified)` for generic HD/FHD mentions, or `Other` otherwise. Overwrites the `Display_Type` column with these more precise labels.

In [ ]:
import re

def extract_display_type(display):

    display = str(display).upper()

    if "DYNAMIC AMOLED" in display:
        return "Dynamic AMOLED"

    elif "SUPER AMOLED" in display:
        return "Super AMOLED"

    elif "AMOLED" in display:
        return "AMOLED"

    elif "POLED" in display:
        return "P-OLED"

    elif "OLED" in display:
        return "OLED"

    elif "IPS LCD" in display:
        return "IPS LCD"

    elif "IPS" in display:
        return "IPS"

    elif "LCD" in display:
        return "LCD"

    elif re.search(r'HD\+|FHD\+|FULL HD\+|FULL HD|HD DISPLAY', display):
        return "LCD (Unspecified)"

    else:
        return "Other"

df["Display_Type"] = df["Display"].apply(extract_display_type)

## Feature Selection
Defines the modeling target (`Price`) and separates the predictor columns into `numerical_features` (RAM, Storage, Battery, Display_Size, Rear_Main_MP, Front_MP, Rear_Camera_Count) and `categorical_features` (Brand, Processor_Clean, Display_Type). Builds the final `X` (features) and `y` (target) datasets.

In [ ]:
# ==========================================
# Feature Selection
# ==========================================

target = "Price"

numerical_features = [
    "RAM",
    "Storage",
    "Battery",
    "Display_Size",
    "Rear_Main_MP",
    "Front_MP",
    "Rear_Camera_Count"
]

categorical_features = [
    "Brand",
    "Processor_Clean",
    "Display_Type"
]

selected_features = numerical_features + categorical_features

X = df[selected_features]

y = df[target]

print(f"Features Selected : {len(selected_features)}")
print(f"Dataset Shape     : {X.shape}")

Features Selected : 10
Dataset Shape     : (5627, 10)


## Train-Test Split
Splits `X` and `y` into training (80%) and testing (20%) sets using a fixed `random_state` for reproducibility.

In [ ]:
# ==========================================
# Train-Test Split
# ==========================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Set :", X_train.shape)
print("Testing Set  :", X_test.shape)

Training Set : (4501, 10)
Testing Set  : (1126, 10)


## Column Transformer
Builds a `ColumnTransformer` that one-hot encodes the categorical features (ignoring unseen categories at inference time) while passing the numerical features through unchanged. This will be the shared preprocessing step for all models.

In [ ]:
# ==========================================
# Column Transformer
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

## Print Feature Lists
Displays the numerical, categorical, and combined selected feature lists as a quick sanity check before modeling.

In [ ]:
print("Numerical Features")
print(numerical_features)

print("\nCategorical Features")
print(categorical_features)

print("\nSelected Features")
print(selected_features)

Numerical Features
['RAM', 'Storage', 'Battery', 'Display_Size', 'Rear_Main_MP', 'Front_MP', 'Rear_Camera_Count']

Categorical Features
['Brand', 'Processor_Clean', 'Display_Type']

Selected Features
['RAM', 'Storage', 'Battery', 'Display_Size', 'Rear_Main_MP', 'Front_MP', 'Rear_Camera_Count', 'Brand', 'Processor_Clean', 'Display_Type']


## Import Pipeline & Metrics
Imports `Pipeline` (to chain preprocessing and modeling steps) and the evaluation metric functions (`mean_absolute_error`, `mean_squared_error`, `r2_score`).

In [ ]:
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Initialize Results Store
Creates an empty `results` dictionary to collect performance metrics for each baseline model trained below.

In [ ]:
results = {}

## Model Evaluation Function
Defines `train_and_evaluate(model, model_name)`, a reusable helper that: (1) builds a `Pipeline` combining the shared preprocessor with the given model, (2) fits it on the training data, (3) predicts on the test set, (4) computes MAE, RMSE, and R², (5) stores the metrics in `results`, and (6) prints a formatted summary. Returns the fitted pipeline.

In [ ]:
# ==========================================
# Model Evaluation Function
# ==========================================

def train_and_evaluate(model, model_name):
    """
    Train a model using the preprocessing pipeline
    and evaluate its performance.
    """

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Prediction
    y_pred = pipeline.predict(X_test)

    # Metrics
    mae = mean_absolute_error(y_test, y_pred)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    r2 = r2_score(y_test, y_pred)

    # Save Results
    results[model_name] = {
        "MAE": round(mae,2),
        "RMSE": round(rmse,2),
        "R2 Score": round(r2,4)
    }

    print("="*50)
    print(model_name)
    print("="*50)

    print(f"MAE       : {mae:.2f}")
    print(f"RMSE      : {rmse:.2f}")
    print(f"R² Score  : {r2:.4f}")

    return pipeline

## Linear Regression
Trains and evaluates a baseline `LinearRegression` model using `train_and_evaluate`.

In [ ]:
# ==========================================
# Linear Regression
# ==========================================

lr_model = LinearRegression()

lr_pipeline = train_and_evaluate(
    lr_model,
    "Linear Regression"
)

Linear Regression
MAE       : 8703.15
RMSE      : 13291.44
R² Score  : 0.8516


## Random Forest
Trains and evaluates a baseline `RandomForestRegressor` (200 trees) using `train_and_evaluate`.

In [ ]:
# ==========================================
# Random Forest
# ==========================================

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_pipeline = train_and_evaluate(
    rf_model,
    "Random Forest"
)

Random Forest
MAE       : 2605.64
RMSE      : 5911.21
R² Score  : 0.9706


## XGBoost
Trains and evaluates a baseline `XGBRegressor` (200 estimators, learning rate 0.1, max depth 6) using `train_and_evaluate`.

In [ ]:
# ==========================================
# XGBoost
# ==========================================

xgb_model = XGBRegressor(

    objective="reg:squarederror",

    n_estimators=200,

    learning_rate=0.1,

    max_depth=6,

    random_state=42,

    n_jobs=-1
)

xgb_pipeline = train_and_evaluate(
    xgb_model,
    "XGBoost"
)

XGBoost
MAE       : 3675.51
RMSE      : 6221.54
R² Score  : 0.9675


## Baseline Model Comparison
Collects the stored `results` for all three baseline models into a DataFrame and sorts them by R² score (descending) to compare their performance at a glance.

In [ ]:
# ==========================================
# Model Comparison
# ==========================================

comparison = (
    pd.DataFrame(results)
      .T
      .sort_values("R2 Score", ascending=False)
)

comparison

,MAE,RMSE,R2 Score
Random Forest,2605.64,5911.21,0.9706
XGBoost,3675.51,6221.54,0.9675
Linear Regression,8703.15,13291.44,0.8516


## Random Forest Parameter Grid (draft)
Defines an initial candidate hyperparameter grid for tuning the Random Forest model. (Superseded by the final grid used later in the notebook.)

In [ ]:
param_distributions = {
    "model__n_estimators":[100,200,300,500],
    "model__max_depth":[None,10,20,30,40],
    "model__min_samples_split":[2,5,10],
    "model__min_samples_leaf":[1,2,4]
}

## XGBoost Parameter Grid (draft)
Defines an initial candidate hyperparameter grid for tuning the XGBoost model. (Superseded by the final grid used later in the notebook.)

In [ ]:
param_distributions = {
    "model__n_estimators":[100,200,300],
    "model__learning_rate":[0.01,0.05,0.1],
    "model__max_depth":[3,5,7],
    "model__subsample":[0.8,1.0],
    "model__colsample_bytree":[0.8,1.0]
}

## Import RandomizedSearchCV
Imports the randomized hyperparameter search tool used for tuning the models.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

## Hyperparameter Tuning Function
Defines `tune_model(model, params, model_name)`, which: (1) wraps the model in the shared preprocessing pipeline, (2) runs `RandomizedSearchCV` with 20 iterations and 5-fold cross-validation optimizing for R², (3) evaluates the best-found model on the held-out test set, and (4) prints the best hyperparameters and performance metrics. Returns the best fitted model and the search object.

In [ ]:
# ==========================================
# Hyperparameter Tuning Function
# ==========================================

def tune_model(model, params, model_name):

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=20,
        cv=5,
        scoring="r2",
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    y_pred = best_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("="*60)
    print(model_name)
    print("="*60)

    print("\nBest Parameters")
    print(search.best_params_)

    print("\nPerformance")

    print(f"MAE  : {mae:.2f}")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.4f}")

    return best_model, search

## Random Forest Parameter Grid (final)
Defines the final hyperparameter search space for Random Forest: number of trees, max depth, min samples to split/leaf, and max features considered per split.

In [ ]:
rf_params = {

    "model__n_estimators":[100,200,300,500],

    "model__max_depth":[
        None,
        10,
        20,
        30,
        40
    ],

    "model__min_samples_split":[
        2,
        5,
        10
    ],

    "model__min_samples_leaf":[
        1,
        2,
        4
    ],

    "model__max_features":[
        "sqrt",
        "log2"
    ]

}

## Tune Random Forest
Runs `tune_model` on a fresh `RandomForestRegressor` using the final parameter grid, storing the best model as `best_rf` and the search object as `rf_search`.

In [ ]:
best_rf, rf_search = tune_model(

    RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),

    rf_params,

    "Random Forest (Tuned)"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Random Forest (Tuned)

Best Parameters
{'model__n_estimators': 300, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_features': 'log2', 'model__max_depth': None}

Performance
MAE  : 3177.75
RMSE : 6544.45
R²   : 0.9640


## XGBoost Parameter Grid (final)
Defines the final hyperparameter search space for XGBoost: number of estimators, learning rate, max depth, subsample ratio, and column subsample ratio.

In [ ]:
xgb_params = {

    "model__n_estimators":[100,200,300],

    "model__learning_rate":[
        0.01,
        0.05,
        0.1
    ],

    "model__max_depth":[
        3,
        5,
        7
    ],

    "model__subsample":[
        0.8,
        1.0
    ],

    "model__colsample_bytree":[
        0.8,
        1.0
    ]

}

## Tune XGBoost
Runs `tune_model` on a fresh `XGBRegressor` using the final parameter grid, storing the best model as `best_xgb` and the search object as `xgb_search`.

In [ ]:
best_xgb, xgb_search = tune_model(

    XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ),

    xgb_params,

    "XGBoost (Tuned)"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
XGBoost (Tuned)

Best Parameters
{'model__subsample': 0.8, 'model__n_estimators': 300, 'model__max_depth': 7, 'model__learning_rate': 0.1, 'model__colsample_bytree': 1.0}

Performance
MAE  : 3156.94
RMSE : 5889.91
R²   : 0.9709


## Tuned Model Comparison
Generates predictions from both tuned models (`best_rf`, `best_xgb`) on the test set, then builds a DataFrame comparing their MAE, RMSE, and R² scores, sorted by R² to identify the best-performing tuned model.

In [ ]:
rf_pred = best_rf.predict(X_test)
xgb_pred = best_xgb.predict(X_test)

comparison_tuned = pd.DataFrame({

    "Model":[
        "Random Forest",
        "XGBoost"
    ],

    "MAE":[
        mean_absolute_error(y_test, rf_pred),
        mean_absolute_error(y_test, xgb_pred)
    ],

    "RMSE":[
        np.sqrt(mean_squared_error(y_test, rf_pred)),
        np.sqrt(mean_squared_error(y_test, xgb_pred))
    ],

    "R2 Score":[
        r2_score(y_test, rf_pred),
        r2_score(y_test, xgb_pred)
    ]

})

comparison_tuned.round(4).sort_values(
    "R2 Score",
    ascending=False
)

,Model,MAE,RMSE,R2 Score
1,XGBoost,3156.9360,5889.9131,0.9709
0,Random Forest,3177.7512,6544.4504,0.9640


## Import joblib
Imports `joblib`, used for serializing the trained model to disk.

In [ ]:
import joblib

## Save Model
Persists the tuned Random Forest model (`best_rf`) to disk as `price_prediction_model.pkl` for later reuse or deployment.

In [ ]:
joblib.dump(best_rf, "price_prediction_model.pkl")

['price_prediction_model.pkl']

## (Empty Cell)
Unused placeholder cell at the end of the notebook.